# Customer Churn Prediction Project
This Jupyter Notebook contains the complete end-to-end Machine Learning pipeline for predicting customer churn in a telecom dataset.

## Project Objective
Analyze telecom customer data (approx. 7,000 customers) to understand the key drivers of customer churn and build predictive models to identify high-risk customers, allowing the business to take proactive retention actions.

## Libraries Used
- **Pandas & NumPy**: Data loading, manipulation, and cleaning
- **Matplotlib & Seaborn**: Data visualization and Exploratory Data Analysis (EDA)
- **Scikit-Learn**: Machine learning model building, validation, and evaluation

### 1. Data Loading & Preprocessing

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests

# Set seaborn style for visual excellence
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 14,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.titlesize': 16
})

# Download the dataset if not present
DATA_URL = "https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv"
DATA_FILE = "data/WA_Fn-UseC_-Telco-Customer-Churn.csv"

if not os.path.exists("data"):
    os.makedirs("data")

if not os.path.exists(DATA_FILE):
    print("Downloading dataset...")
    r = requests.get(DATA_URL)
    with open(DATA_FILE, "wb") as f:
        f.write(r.content)
    print("Downloaded successfully.")
else:
    print("Dataset already exists locally.")

# Load data
df = pd.read_csv(DATA_FILE)
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")

In [ ]:
# Display dataset types and details
df.info()

In [ ]:
# Display first 5 rows
df.head()

### 2. Data Cleaning & Cleaning Gotchas
- **TotalCharges** is loaded as an `object` type because it contains blank space values (`" "`) for new customers with tenure = 0.
- We will drop the `customerID` column as it does not contain predictive power.
- We will map the target `Churn` column to 1 (Yes) and 0 (No).

In [ ]:
# Drop customerID
df_clean = df.drop("customerID", axis=1, errors='ignore')

# Convert TotalCharges to numeric, coercing spaces to NaN
df_clean["TotalCharges"] = pd.to_numeric(df_clean["TotalCharges"], errors='coerce')

# Check for missing values
print("Missing values in TotalCharges:", df_clean["TotalCharges"].isnull().sum())

# Fill missing values with 0.0 (since tenure is 0, they haven't been billed yet)
df_clean["TotalCharges"] = df_clean["TotalCharges"].fillna(0.0)

# Map target Churn variable
df_clean["Churn"] = df_clean["Churn"].map({"Yes": 1, "No": 0})

print("Data cleaning completed successfully.")

### 3. Exploratory Data Analysis (EDA)
Let's perform comprehensive EDA to understand customer demographics, contract details, payments, and how they relate to churn.

In [ ]:
# Colors for plotting
colors = ["#2A9D8F", "#E76F51"]

# Churn Distribution
plt.figure(figsize=(6, 4))
ax = sns.countplot(x="Churn", data=df_clean, palette=colors)
plt.title("Customer Churn Distribution")
plt.xticks([0, 1], ["Retained", "Churned"])

total = len(df_clean)
for p in ax.patches:
    height = p.get_height()
    percentage = (height / total) * 100
    ax.annotate(f'{height}\n({percentage:.1f}%)', 
                (p.get_x() + p.get_width() / 2., height - (height * 0.15)),
                ha='center', va='center', color='white', fontweight='bold')
plt.tight_layout()
plt.show()

**Insight:** The dataset exhibits class imbalance: ~26.5% of customers have churned, and ~73.5% were retained. Metrics like Precision, Recall, and F1-score will be crucial for evaluation.

In [ ]:
# Churn vs Contract Type
plt.figure(figsize=(8, 4))
sns.countplot(x="Contract", hue="Churn", data=df_clean, palette=colors)
plt.title("Churn Distribution by Contract Type")
plt.legend(title="Status", labels=["Retained", "Churned"])
plt.tight_layout()
plt.show()

**Insight:** Month-to-month contract customers have a significantly higher churn risk. Migrating customers to 1-year or 2-year contracts should be a high priority retention strategy.

In [ ]:
# Churn vs Tenure
plt.figure(figsize=(9, 4.5))
sns.kdeplot(x="tenure", hue="Churn", data=df_clean, fill=True, palette=colors, common_norm=False, alpha=0.5)
plt.title("Tenure Density Distribution by Churn Status")
plt.xlabel("Tenure (Months)")
plt.legend(title="Status", labels=["Churned", "Retained"])
plt.tight_layout()
plt.show()

**Insight:** Churn is highly concentrated in the first 12 months. Early tenure customer onboarding and support check-ins are critical.

In [ ]:
# Churn vs Monthly Charges
plt.figure(figsize=(9, 4.5))
sns.kdeplot(x="MonthlyCharges", hue="Churn", data=df_clean, fill=True, palette=colors, common_norm=False, alpha=0.5)
plt.title("Monthly Charges Density Distribution by Churn Status")
plt.xlabel("Monthly Charges ($)")
plt.legend(title="Status", labels=["Churned", "Retained"])
plt.tight_layout()
plt.show()

**Insight:** Customers with higher monthly charges ($70 - $100+) show a higher propensity to churn. Price sensitivity is a major concern.

In [ ]:
# Churn vs Payment Method
plt.figure(figsize=(10, 5))
sns.countplot(y="PaymentMethod", hue="Churn", data=df_clean, palette=colors)
plt.title("Churn Distribution by Payment Method")
plt.legend(title="Status", labels=["Retained", "Churned"])
plt.tight_layout()
plt.show()

**Insight:** Electronic check users have a significantly higher churn rate than customers using mailed checks or automatic payments (credit card or bank transfer). Moving electronic check users to auto-pay is recommended.

In [ ]:
# Correlation Heatmap
df_corr = df_clean[["tenure", "MonthlyCharges", "TotalCharges", "Churn"]].copy()
plt.figure(figsize=(6, 5))
sns.heatmap(df_corr.corr(), annot=True, cmap="coolwarm", fmt=".2f", square=True)
plt.title("Correlation Matrix of Numerical Features & Churn")
plt.tight_layout()
plt.show()

**Insight:** Tenure is negatively correlated with Churn (-0.35), showing long-term customers are loyal. TotalCharges is highly correlated with Tenure (0.83), causing potential multicollinearity.

In [ ]:
# Distribution plots for numerical features
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
sns.histplot(df_clean["tenure"], kde=True, ax=axes[0], color="#264653")
axes[0].set_title("Tenure Distribution")
sns.histplot(df_clean["MonthlyCharges"], kde=True, ax=axes[1], color="#2a9d8f")
axes[1].set_title("Monthly Charges Distribution")
sns.histplot(df_clean["TotalCharges"], kde=True, ax=axes[2], color="#e76f51")
axes[2].set_title("Total Charges Distribution")
plt.suptitle("Feature Distributions")
plt.tight_layout()
plt.show()

In [ ]:
# Customer retention patterns across categories (SeniorCitizen, InternetService, TechSupport)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.countplot(x="SeniorCitizen", hue="Churn", data=df_clean, ax=axes[0], palette=colors)
axes[0].set_xticklabels(["Non-Senior", "Senior"])
axes[0].set_title("Retention by Senior Citizen Status")

sns.countplot(x="InternetService", hue="Churn", data=df_clean, ax=axes[1], palette=colors)
axes[1].set_title("Retention by Internet Service")

sns.countplot(x="TechSupport", hue="Churn", data=df_clean, ax=axes[2], palette=colors)
axes[2].set_title("Retention by Tech Support Availability")
plt.tight_layout()
plt.show()

**Insight:** Fiber optic customers have significantly higher churn rates. Tech support availability is a major factor in retaining customers.

### 4. Feature Engineering & Preprocessing
We will encode categorical variables using One-Hot Encoding and scale numerical features using `StandardScaler`.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Split into features and target
X = df_clean.drop("Churn", axis=1)
y = df_clean["Churn"]

# Encode categorical features
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

# Train-test split (80/20) with stratified sampling
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

# Scale numerical features
numerical_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test_scaled[numerical_cols] = scaler.transform(X_test[numerical_cols])

print(f"Preprocessed shapes:\n  X_train: {X_train_scaled.shape}\n  X_test: {X_test_scaled.shape}")

### 5. Model Building & Cross-Validation

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold

# Define models
lr_model = LogisticRegression(max_iter=1000, random_state=42)
rf_model = RandomForestClassifier(n_estimators=150, max_depth=8, min_samples_leaf=4, random_state=42, n_jobs=-1)

# 5-Fold Stratified Cross-Validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lr_cv_scores = cross_val_score(lr_model, X_train_scaled, y_train, cv=cv, scoring='accuracy')
rf_cv_scores = cross_val_score(rf_model, X_train_scaled, y_train, cv=cv, scoring='accuracy')

print(f"Logistic Regression Mean CV Accuracy: {lr_cv_scores.mean():.4f} (+/- {lr_cv_scores.std():.4f})")
print(f"Random Forest Mean CV Accuracy:      {rf_cv_scores.mean():.4f} (+/- {rf_cv_scores.std():.4f})")

# Fit models on the full training set
lr_model.fit(X_train_scaled, y_train)
rf_model.fit(X_train_scaled, y_train)

### 6. Model Evaluation
Let's evaluate the models on the test set using Accuracy, Precision, Recall, F1, and ROC-AUC.

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, roc_curve
)

models = {
    "Logistic Regression": lr_model,
    "Random Forest": rf_model
}

metrics_list = []
for name, model in models.items():
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    
    metrics_list.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })

df_metrics = pd.DataFrame(metrics_list)
df_metrics

In [ ]:
# Plot Confusion Matrices side-by-side
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for i, (name, model) in enumerate(models.items()):
    y_pred = model.predict(X_test_scaled)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[i], cbar=False,
                xticklabels=["Retained", "Churned"], yticklabels=["Retained", "Churned"])
    axes[i].set_title(f"{name} Confusion Matrix")
    axes[i].set_xlabel("Predicted")
    axes[i].set_ylabel("True")
plt.tight_layout()
plt.show()

In [ ]:
# Plot ROC Curves
plt.figure(figsize=(8, 6))
for name, model in models.items():
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {roc_auc_score(y_test, y_prob):.3f})")
plt.plot([0, 1], [0, 1], 'k--', label="Random Guessing (AUC = 0.500)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves Comparison")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.7)
plt.show()

In [ ]:
# Feature Importance for Random Forest
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]
feature_names = X_train_scaled.columns.tolist()
top_n = 15

plt.figure(figsize=(10, 6))
sns.barplot(x=importances[indices[:top_n]], y=np.array(feature_names)[indices[:top_n]], palette="viridis")
plt.title("Top 15 Feature Importances (Random Forest)")
plt.xlabel("Relative Importance Score")
plt.show()

### 7. Summary of Findings & Actionable Retention Strategies
1. **Contract Strategy**: Month-to-month contracts are the strongest predictor of churn. Migrations to 1 or 2-year contracts through promotions can drop churn drastically.
2. **Onboarding focus**: High customer churn occurs within the first 12 months. Proactive support in the early customer journey can preserve customer lifetime value.
3. **Value-added Services**: Offering Online Security and Tech Support decreases churn significantly. Promoting these features can stabilize retention.